# C-PROOF glider data for BarkleyScope, from ERDDAP

This notebook is a walkthrough of the glider data pipeline behind the BarkleyScope
dashboard. It finds [C-PROOF](https://cproof.uvic.ca/) glider deployments on the
[IOOS Glider DAC ERDDAP](https://gliders.ioos.us/erddap/index.html) that entered our
study box off the west coast of Vancouver Island, downloads what they measured there,
and keeps it in a netCDF archive that a scheduled job tops up every day.

The steps are:

1. **Discover** — ask ERDDAP's *advanced search* which C-PROOF datasets overlap the box.
2. **Download** — pull only the observations inside the box, per deployment.
3. **Quality control** — a gross-range screen, because real-time feeds are uncalibrated.
4. **Archive** — append to a netCDF file, resuming from what is already stored.
5. **Read and plot** — tracks on a map, and temperature through the water column.

Every step is a function in [`cproof_glider.py`](cproof_glider.py), which this notebook
imports rather than redefines. The daily [GitHub Action](../.github/workflows/update-glider-archive.yml)
imports the same module, so the notebook and the scheduled job cannot drift apart —
a GitHub Action cannot import functions defined in notebook cells.

> If you only want to *use* the data, you do not need any of this. Jump to
> [section 5](#5.-Read-the-archive-back) — `read_archive()` is the whole interface.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The pipeline lives next to this notebook, in data/.
sys.path.insert(0, str(Path.cwd() if Path("cproof_glider.py").exists() else Path("data")))
import cproof_glider as cproof

print(f"pandas {pd.__version__} | module: {Path(cproof.__file__).resolve()}")

## 1. Configuration

Everything that defines *what* gets pulled lives in `cproof_glider.py`, not in this
notebook — one definition, shared by the notebook and the scheduled job.

The study box is a single rectangle, which is convenient: ERDDAP can apply it
server-side exactly, so there is no client-side spatial filtering to get wrong.

We ask for **all seven science variables** C-PROOF gliders fly, not just the one
currently being plotted. The download is dominated by round trips rather than by
columns, so carrying all seven means adding a parameter to the dashboard later is a
plotting change instead of a re-harvest.

In [ ]:
(lon_min, lon_max), (lat_min, lat_max) = cproof.BOX["lon"], cproof.BOX["lat"]

print(f"Server      : {cproof.ERDDAP}")
print(f"Institution : {cproof.INSTITUTION}")
print(f"Study box   : lon {lon_min} to {lon_max}, lat {lat_min} to {lat_max}")
print(f"Record from : {cproof.START_OF_RECORD}")
print(f"Variables   : {', '.join(cproof.SCIENCE_VARS)}")
print()
for mode in ("realtime", "delayed"):
    path = cproof.archive_path(mode)
    size = f"{path.stat().st_size / 1e6:.1f} MB" if path.exists() else "not built yet"
    print(f"{mode:9s} archive: {path.name}  ({size})")

### If the delayed archive is missing

It is ~37 MB, so it is not carried in git. Build it once — it takes a couple of
minutes — and then it is yours:

```bash
python data/update_cproof_glider.py --mode delayed
```

The real-time archive *is* in git, because the daily job commits it back.

## 2. Discover which deployments reached the box

The advanced-search endpoint filters on each dataset's *overall* bounding box and time
range, which narrows the DAC's ~1000 glider datasets down to the handful worth
downloading. `find_datasets()` then confirms the institution attribute, because
`searchFor` is a free-text match and will happily match a mention in an abstract.

Real-time and delayed-mode datasets are separated here by their `-delayed` suffix,
because they are different records and we keep them in different archives — see
section 6.

In [ ]:
live = cproof.find_datasets(mode="realtime", start_time=cproof.START_OF_RECORD)
print(f"{len(live)} real-time C-PROOF deployment(s) overlap the study box\n")
live[["datasetID", "glider", "minTime", "maxTime"]].head(10)

**Do not be surprised if that number moves.** The DAC catalogue is not stable: repeated
identical searches have returned 10, 25, and 38 datasets within minutes as the server
reloads datasets. Everything downstream is built to tolerate it — updates only ever
*add*, so a deployment the search misses now is picked up by the next run, and
successive runs converge on full coverage.

## 3. What one deployment looks like

Before harvesting everything, pull a single deployment so the shape of the data is
concrete. `fetch_dataset()` asks ERDDAP only for the variables that deployment actually
carries — roughly a third of these gliders fly no oxygen optode, and requesting a
variable a dataset does not have is an error, not an empty column. Missing sensors come
back as `NaN` so that every frame has the same columns whatever the glider carried.

In [ ]:
if len(live):
    example_id = live.datasetID.iloc[0]
    print(f"{example_id} carries: {', '.join(cproof.science_variables(example_id))}\n")
    print(cproof.build_url(example_id, start_time="2026-01-01T00:00:00Z")[:220], "...\n")

    example = cproof.fetch_dataset(example_id, start_time="2026-01-01T00:00:00Z")
    print(f"{len(example):,} observations inside the box")
    display(example.head())
else:
    print("No real-time deployments found right now — the archive below still works.")

## 4. Update the archive

This is the same call the daily job makes. It is worth understanding what it does,
because the behaviour is the whole reason the pipeline is reliable:

- It reads the **high-water mark** for each deployment — the latest observation already
  stored — straight out of the archive, not from a sidecar state file. State that lives
  in the data cannot drift out of sync with the data.
- It fetches each deployment only from that mark onward, then appends.

So it is **idempotent** (run it twice, the second run appends nothing) and
**self-healing** (run it after a missed week and it backfills the whole gap, not just
the last day). Quality control happens on the way in: values outside a plausible range
are *blanked, not dropped*, so a failed optode does not cost that observation its
temperature.

In [ ]:
summary = cproof.update_archive(mode="realtime")
summary

Run that cell a second time and it will report `Appended 0` — everything it found, it
already has. That is the property that makes it safe to schedule.

## 5. Read the archive back

`read_archive()` is the entire interface for anyone building a plot or a dashboard. It
returns a tidy DataFrame, one row per observation, sorted by time.

Use `variables=` on the delayed archive: reading all seven columns across three million
observations is several hundred megabytes in memory, and most plots need one or two.

In [ ]:
recent = cproof.read_archive(cproof.REALTIME_ARCHIVE, last_days=7)     # the dashboard call
whole = cproof.read_archive(cproof.REALTIME_ARCHIVE)

print(f"Last 7 days : {len(recent):,} observations")
print(f"Whole file  : {len(whole):,} observations, "
      f"{whole.time.min():%Y-%m-%d} to {whole.time.max():%Y-%m-%d}, "
      f"{whole.glider.nunique()} glider(s)")
whole.head()

In [ ]:
# Coverage matters more than row counts: a third of deployments fly no optode, so an
# oxygen panel will simply be empty for them. Always check before plotting.
coverage = (whole[cproof.SCIENCE_VARS].notna().mean() * 100).round(1)
print("Percentage of observations carrying each variable:")
print(coverage.to_string())

print("\nBy deployment:")
print(whole.groupby("deployment").agg(
    observations=("time", "size"),
    first=("time", "min"),
    last=("time", "max"),
    max_depth=("depth", "max"),
).to_string())

### A note on negative chlorophyll and backscatter

They are not errors. The optical channels are raw counts converted with factory
coefficients and are **not dark-corrected**, so small negative values are ordinary
instrument behaviour in clear deep water — the 1st percentile of chlorophyll here is
about -0.46 mg m⁻³. Clipping at zero would blank roughly a quarter of the bio-optical
record. If you want a presentation-friendly axis, clamp the *colour scale*, not the data.

## 6. Real-time is not delayed-mode

The two archives are kept separate rather than blended, because the difference between
them is scientific rather than cosmetic:

- **Delayed-mode is calibrated. Real-time is not.** Real-time values have had only the
  gross-range screen above applied. Do not present them as calibrated measurements.
- **Delayed-mode is far denser** — the real-time feed is heavily decimated to save
  satellite bandwidth.
- **They cover different periods.** Reprocessing happens after the glider is recovered,
  so delayed-mode data lag real time by months to years.

A reasonable dashboard shows real-time for *what is happening now* and delayed-mode for
*what is normal for this time of year*, and labels which is which.

In [ ]:
if cproof.DELAYED_ARCHIVE.exists():
    history = cproof.read_archive(cproof.DELAYED_ARCHIVE, variables=["temperature"])
    print(f"Delayed archive: {len(history):,} observations, "
          f"{history.time.min():%Y-%m-%d} to {history.time.max():%Y-%m-%d}")

    # A delayed dataset ID is the real-time ID plus a "-delayed" suffix, so strip it to
    # line the two records up on the same deployment.
    base = history.deployment.str.removesuffix("-delayed")
    both = sorted(set(base) & set(whole.deployment))
    if both:
        counts = pd.DataFrame([{"deployment": name,
                                "realtime": int((whole.deployment == name).sum()),
                                "delayed": int((base == name).sum())} for name in both])
        counts["delayed per real-time obs"] = (counts.delayed / counts.realtime).round(0)
        print("\nThe same deployment, seen through both feeds:")
        print(counts.to_string(index=False))
    else:
        print("\nNo deployment is in both archives yet — reprocessing lags real time "
              "by months to years, so the overlap comes and goes.")
else:
    history = None
    print("Delayed archive not built yet — run:  python data/update_cproof_glider.py --mode delayed")


## 7. Where did the gliders go?

Map the tracks against the study box to confirm the spatial filter did what we expect —
every point should sit inside the rectangle, because ERDDAP applied the constraint
server-side.

In [ ]:
def study_area_axes(figsize=(9, 8)):
    """A map axes over the study box, with coastlines if cartopy can fetch them."""
    extent = [lon_min - 0.3, lon_max + 0.3, lat_min - 0.2, lat_max + 0.2]
    try:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature

        fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": ccrs.PlateCarree()})
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.GSHHSFeature(scale="high"), facecolor="0.85", edgecolor="0.4")
        gl = ax.gridlines(draw_labels=True, alpha=0.3)
        gl.top_labels = gl.right_labels = False
        return fig, ax, ccrs.PlateCarree()
    except Exception as error:                     # no cartopy, or no coastline download
        print(f"Plain axes (cartopy unavailable: {type(error).__name__})")
        fig, ax = plt.subplots(figsize=figsize)
        ax.set_xlim(extent[:2]); ax.set_ylim(extent[2:])
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        return fig, ax, None


fig, ax, crs = study_area_axes()
transform = {"transform": crs} if crs else {}

ax.add_patch(plt.Rectangle((lon_min, lat_min), lon_max - lon_min, lat_max - lat_min,
                           fill=False, edgecolor="crimson", lw=2, zorder=5,
                           label="BarkleyScope box", **transform))

for deployment, track in whole.groupby("deployment"):
    ax.plot(track.longitude, track.latitude, ".", ms=2, alpha=0.6, label=deployment, **transform)

ax.legend(loc="upper left", fontsize=8, markerscale=4)
ax.set_title(f"C-PROOF glider tracks in the BarkleyScope box\n"
             f"{len(whole):,} real-time observations")
plt.show()

## 8. Depth sections

Colour every observation by its value against time and depth to see the water-column
structure the gliders sampled. Each vertical stripe is one dive.

In [ ]:
PANELS = [("temperature", "Temperature (°C)", "viridis"),
          ("salinity", "Salinity (PSU)", "cividis"),
          ("oxygen_concentration", "Oxygen (µmol L$^{-1}$)", "magma"),
          ("chlorophyll", "Chlorophyll (mg m$^{-3}$)", "YlGn")]

panels = [p for p in PANELS if whole[p[0]].notna().any()]
fig, axes = plt.subplots(len(panels), 1, figsize=(12, 3.4 * len(panels)),
                         sharex=True, squeeze=False)

for ax, (variable, label, cmap) in zip(axes[:, 0], panels):
    have = whole[whole[variable].notna()]
    # Clamp the colour scale rather than the data, so outliers do not flatten the range.
    low, high = have[variable].quantile([0.02, 0.98])
    points = ax.scatter(have.time, have.depth, c=have[variable], s=3,
                        cmap=cmap, vmin=low, vmax=high)
    fig.colorbar(points, ax=ax, label=label, pad=0.01)
    ax.invert_yaxis()
    ax.set_ylabel("Depth (m)")
    ax.set_title(f"{label} — {have[variable].notna().sum():,} observations", fontsize=10)

axes[-1, 0].set_xlabel("Time (UTC)")
fig.suptitle("Real-time glider observations in the BarkleyScope box", y=0.995)
fig.tight_layout()
plt.show()

### The same water column, calibrated

If the delayed archive is built, the contrast is worth seeing: the same box, the same
gliders, reprocessed after recovery. Note the density of sampling as much as the values.

In [ ]:
if history is not None and len(history):
    fig, ax = plt.subplots(figsize=(12, 4))
    low, high = history.temperature.quantile([0.02, 0.98])
    points = ax.scatter(history.time, history.depth, c=history.temperature, s=1,
                        cmap="viridis", vmin=low, vmax=high)
    fig.colorbar(points, ax=ax, label="Temperature (°C)", pad=0.01)
    ax.invert_yaxis()
    ax.set_ylabel("Depth (m)")
    ax.set_xlabel("Time (UTC)")
    ax.set_title(f"Delayed-mode temperature, {len(history):,} observations "
                 f"(calibrated, quality controlled)")
    plt.show()
else:
    print("Build the delayed archive to see this panel.")

## 9. Dynamic data access — keeping it current

The payoff of hitting ERDDAP programmatically is picking up new observations as they
land. The naive version of this is a polling loop in a notebook, which only runs while
the notebook is open — and this JupyterHub pod is culled when idle.

So the schedule lives in [GitHub Actions](../.github/workflows/update-glider-archive.yml)
instead. At 00:00 UTC it runs exactly one command:

```bash
python data/update_cproof_glider.py --mode realtime
```

which calls the same `update_archive()` from section 4, verifies the result, and commits
the archive back. The cell below is that job, run by hand — and because updates are
idempotent, running it right after section 4 should append **nothing**.

In [ ]:
again = cproof.update_archive(mode="realtime")
print(f"\nAppended on the second run: {again['appended']}  "
      f"(0 is the expected, correct answer)")

marks = cproof.high_water_marks(cproof.REALTIME_ARCHIVE)
print("\nHigh-water mark per deployment — the archive's own state, read from the data:")
for deployment, mark in sorted(marks.items()):
    print(f"  {deployment:32s} {mark}")

## 10. What next

- `data/README.md` documents the archives, the variables, and their caveats.
- `read_archive()` is all a plot needs — `last_days=7` for the live view,
  `variables=["temperature"]` for the historical one.
- The bonus section below builds a streamlit dashboard on top of the same data.

# Bonus Part!! 
## Creating an online data dashboard

Please only complete this section if you have extra time in the workshop or during post-workshop time. 

This section will walk through using ERDDAP data access protocols and a package called `streamlit` to create an online data dashboard. `streamlit` enables developers to create interactive online data apps in python without writing javascript or HTML code. 

Although we are writing the code to run our dashboard within this notebook, the dashboard code is run independently from the cells above. As a result, variables defined in the cells above will not be transferred to the dashboard and will need to be redefined in the dashboard code. This means we will have to recreate our ERDDAP data call within the dashboard code. Now that you have experience writing an ERDDAP URL, this section of the code has been left blank for you to fill in yourself!
> **Heads up:** the dashboard section below is still the original workshop example built
> against the ONC ERDDAP (`oxygen_corrected`, temperature in Kelvin). To point it at
> C-PROOF, reuse `build_url()` / `fetch_dataset()` from the cells above.


In [ ]:
%pip show streamlit 

# If not installed, you will get a response: WARNING: Package(s) not found: streamlit
# Note: you may need to restart the kernel to use updated packages.

# If not installed run the cell below. If installed you may skip it. 

In [ ]:
%pip show plotly

# If not installed, you will get a response: WARNING: Package(s) not found: plotly
# Note: you may need to restart the kernel to use updated packages.

# If not installed run the cell below. If installed you may skip it. 

In [ ]:
# Skip if streamlit already installed.
%pip install streamlit

In [ ]:
# Skip if streamlit already installed.
%pip install plotly

In [ ]:
import streamlit as st

## Writing the dashboard code
The code that creates the dashboard will be run from an independent .py file. 
We will create and write that .py file here within this notebook in the cell below.

To start let's keep it simple and let's just give the dashboard a name, 'China Creek Ocean Conditions', and a message, 'Hello from our ERDDAP dashbaord'.

In [ ]:
# Creating the dashboard script
from pathlib import Path

dashboard_code = """
import streamlit as st

st.title("China Creek Ocean Conditions")

st.write("Hello from our ERDDAP dashboard!")
"""

dashboard_path = Path("china_creek_dashboard.py")
dashboard_path.write_text(dashboard_code)

print(f"Dashboard file created: {dashboard_path.resolve()}")

### Check the new file

Look at your jupyter hub home, do you see a new file there? Check its contents, does it match what we wrote in the cell above?

In [ ]:
# What is in the new py file? 
print(dashboard_path.read_text())

### Launch the dashboard

Now we have a .py file that will create our online dashboard. They dashboard will not exist until we execute that .py file. To launch our new dashbaord, execute the .py file in terminal. 
1. Open a new terminal shell separete from the one running this notebook
2. Ensure the working directory matches the current project directory
3. Run the following line: streamlit run china_creek_dashboard.py

You should see something like, "Local URL: http://localhost:8501" returned. 
This is the url of the dashboard. Copy and paste into your web browser. 

### Updating the dashboard code

The dashboard code is currently pretty empty. Here, we will rewrite the china_creek_dashboard.py code we defined above as 'dashboard_code'. The code we write here will be very similar to what we wrote above in cell 2 and 3 of this notebook. Our workflow will be:
1. Create a function to iteratively retrieve data from the ERDDAP and update figure
2. Set up the dashboard
3. Create a static figure placeholder with three traces: O2, temperature, and our hypoxia threshold line
4. Call the function created in step 1

Everything has been filled in for you, except the part of the function where we define the ERDDAP url. Use the example from Cell 2 and 3 to complete this part on your own. To update the dashboard, run the cell below and then refresh your dashboard page. 
Note: A function can be defined at any time, the important thing is where/when the function is *called* within the code. I placed the function at the top so it would be easier to find the section where we define the ERDDAP url that you need to edit. 

In [ ]:
# Add dynamic ERDDAP data plotting to our dashboard code.

from pathlib import Path

dashboard_code = """
import streamlit as st
import pandas as pd
import requests
import io
import time
import plotly.graph_objects as go

# ---------------------------------------------------------
# 1. Function to retrieve and update the data
# ---------------------------------------------------------

@st.fragment(run_every="10s")
def update_dashboard():

    # -----------------------------------------------------
    # Set the time range for the initial request
    # -----------------------------------------------------

    time_series_length = 1  # days, restricted to 1 because ERDDAP is slow

    start_time = time.strftime("%Y-%m-%dT%H:%M:%S.000Z", time.gmtime(time.time() - time_series_length * 24 * 3600))

    # -----------------------------------------------------
    # Construct ERDDAP URL <----- YOUR EDITS HERE
    # -----------------------------------------------------

    XXX
    
    url = (f"{onc_erddap}{dataset_id}{format_ext}?{query_vars}&time>={start_time}")

    # -----------------------------------------------------
    # Get data from ERDDAP
    # -----------------------------------------------------

    try:

        response = requests.get(url, verify=False,timeout=120)
        response.raise_for_status()
        data = pd.read_csv(io.StringIO(response.text))

    except Exception as e:
        st.error(f"Unable to retrieve ERDDAP data: {e}")
        return

    # -----------------------------------------------------
    # Process ERDDAP data
    # -----------------------------------------------------

    # First row contains units
    data_values = data.iloc[1:].copy()

    data_values["time"] = pd.to_datetime(data_values["time"])
    data_values["temperature"] = (pd.to_numeric(data_values["temperature"]) - 273.15)
    data_values["oxygen_corrected"] = pd.to_numeric(data_values["oxygen_corrected"])

    # Sort by time
    data_values = data_values.sort_values("time").reset_index(drop=True)

    # -----------------------------------------------------
    # Determine which data are new
    # -----------------------------------------------------

    if st.session_state.last_time is None:

        # First request: use all available data
        new_values = data_values.copy()

    else:

        # Subsequent requests: only use observations from latest call
        new_values = data_values[data_values["time"] > st.session_state.last_time].copy()

    # -----------------------------------------------------
    # Add new observations to the existing plot
    # -----------------------------------------------------

    if not new_values.empty:

        st.session_state.plot_data = pd.concat(
            [st.session_state.plot_data, new_values[
                ["time", "temperature", "oxygen_corrected"]
            ]], ignore_index=True)

        # Remove duplicate observations
        st.session_state.plot_data = (
            st.session_state.plot_data
            .drop_duplicates(subset="time")
            .sort_values("time")
            .reset_index(drop=True)
        )

        # Update most recent timestamp
        st.session_state.last_time = (new_values["time"].max())

        # -------------------------------------------------
        # Update the existing Plotly traces
        # -------------------------------------------------

        fig = st.session_state.plot_fig

        plot_data = st.session_state.plot_data

        # Temperature trace
        fig.data[0].x = plot_data["time"]
        fig.data[0].y = plot_data["temperature"]

        # Oxygen trace
        fig.data[1].x = plot_data["time"]
        fig.data[1].y = plot_data["oxygen_corrected"]

        # Hypoxia threshold
        fig.data[2].x = [plot_data["time"].min(),plot_data["time"].max()]
        fig.data[2].y = [2 / 1.43, 2 / 1.43]

        # Save the updated figure
        st.session_state.plot_fig = fig

    # -----------------------------------------------------
    # Update latest-value displays
    # -----------------------------------------------------

    if not data_values.empty:

        latest = data_values.iloc[-1]

        temperature_display.metric("Temperature", f"{latest['temperature']:.4f} °C")
        oxygen_display.metric("Oxygen", f"{latest['oxygen_corrected']:.4f} mL/L")
        depth_display.metric("Depth", f"{float(latest['depth']):.1f} m")

        st.caption(f"Most recent observation: {latest['time']} UTC")
        st.caption(f"ERDDAP Query: {url}")

        plot_placeholder.plotly_chart(st.session_state.plot_fig,
            use_container_width=True, key="china_creek_plot")

        # End of function update_dashboard

# ---------------------------------------------------------
# 2. Dashboard settings
# ---------------------------------------------------------

st.set_page_config(page_title="China Creek Ocean Conditions",layout="wide")

st.title("China Creek Ocean Conditions")

col1, col2, col3 = st.columns(3)

temperature_display = col1.empty()
oxygen_display = col2.empty()
depth_display = col3.empty()

plot_placeholder = st.empty()

# ---------------------------------------------------------
# 3. Create the plot ONCE
# ---------------------------------------------------------

if "plot_fig" not in st.session_state:

    fig = go.Figure()

    # Temperature trace
    fig.add_trace(
        go.Scatter(
            x=[],
            y=[],
            name="Temperature",
            mode="lines+markers",
            marker=dict(size=4),
            line=dict(color="blue"),
            yaxis="y1",
            showlegend=True
        )
    )

    # Oxygen trace
    fig.add_trace(
        go.Scatter(
            x=[],
            y=[],
            name="Oxygen",
            mode="lines+markers",
            marker=dict(size=4),
            line=dict(color="red"),
            yaxis="y2",
            showlegend=True
        )
    )

    # Hypoxia threshold
    fig.add_trace(
        go.Scatter(
            x=[],
            y=[],
            name="Hypoxia threshold",
            mode="lines",
            line=dict(dash="dash",color="red"),
            yaxis="y2",
            showlegend=True
        )
    )

    # Figure layout function
    fig.update_layout(

        title="China Creek O₂ Time Series",

        xaxis=dict(title="Time (UTC)"),
        yaxis=dict(title="Temperature (°C)",side="left"),
        yaxis2=dict(title="Oxygen (mL/L)",side="right",overlaying="y"),

        hovermode="x unified",

        height=550,

        # Keep the Plotly chart from resetting 
        uirevision="china-creek"
    )

    st.session_state.plot_fig = fig

    # Store all observations used by the plot
    st.session_state.plot_data = pd.DataFrame(columns=["time","temperature","oxygen_corrected"])

    # Keep track of the most recent observation
    st.session_state.last_time = None

# ---------------------------------------------------------
# Start the dashboard
# ---------------------------------------------------------

update_dashboard()
"""

dashboard_path = Path("china_creek_dashboard.py")
dashboard_path.write_text(dashboard_code)

print(f"Dashboard updated: {dashboard_path.resolve()}")

### Modifying our code to improve plotting speed. 

Notice how slowly the dashboard is plotting? Why might that be? How can we improve the plotting speed?

Hint: What do we know about the speed of data extraction from ERDDAP?
Look at our original notebook Cells 2 & 3, how much data do we initially extract vs. how much do we extract in each loop iteration? How can we modify our code above to only pull *new* data from the ERDDAP rather than pulling a full 24 hrs worth of data each run?

In [ ]:
# Add incremental ERDDAP data requests to our dashboard code.

from pathlib import Path

dashboard_code = """
import streamlit as st
import pandas as pd
import requests
import io
import time
import plotly.graph_objects as go

# ---------------------------------------------------------
# 1. Function to retrieve and update the data
# ---------------------------------------------------------

@st.fragment(run_every="10s")
def update_dashboard():

    # -----------------------------------------------------
    # Set the start time for the ERDDAP request <----- YOUR EDITS HERE
    # -----------------------------------------------------

        start_time = XXX

    # -----------------------------------------------------
    # Construct ERDDAP URL 
    # -----------------------------------------------------

    onc_erddap = "https://dap.oceannetworks.ca/erddap/tabledap/"
    dataset_id = "scalar_1217325"
    format_ext = ".csv"
    query_vars = "time,temperature,oxygen_corrected,oxygen_uncorrected,latitude,longitude,depth"

    url = f"{onc_erddap}{dataset_id}{format_ext}?{query_vars}&time>={start_time}"

    # -----------------------------------------------------
    # Get data from ERDDAP
    # -----------------------------------------------------

    try:
        response = requests.get(url, verify=False, timeout=120)
        response.raise_for_status()
        data = pd.read_csv(io.StringIO(response.text))

    except Exception as e:
        st.error(f"Unable to retrieve ERDDAP data: {e}")
        return

    # -----------------------------------------------------
    # Process ERDDAP data
    # -----------------------------------------------------

    # First row contains units
    data_values = data.iloc[1:].copy()

    data_values["time"] = pd.to_datetime(data_values["time"])
    data_values["temperature"] = pd.to_numeric(data_values["temperature"]) - 273.15
    data_values["oxygen_corrected"] = pd.to_numeric(data_values["oxygen_corrected"])

    # Sort by time
    data_values = data_values.sort_values("time").reset_index(drop=True)

    # -----------------------------------------------------
    # Add new observations to the existing plot
    # -----------------------------------------------------

    if not data_values.empty:

        st.session_state.plot_data = pd.concat(
            [st.session_state.plot_data, data_values[["time", "temperature", "oxygen_corrected"]]],
            ignore_index=True
        )

        # Remove duplicate observations
        st.session_state.plot_data = st.session_state.plot_data.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)

        # Update start time for ERDDAP
        st.session_state.start_time = data_values["time"].iloc[-1].strftime("%Y-%m-%dT%H:%M:%S.%fZ")
        
        # -------------------------------------------------
        # Update the existing Plotly traces
        # -------------------------------------------------

        fig = st.session_state.plot_fig
        plot_data = st.session_state.plot_data

        # Temperature trace
        fig.data[0].x = plot_data["time"]
        fig.data[0].y = plot_data["temperature"]

        # Oxygen trace
        fig.data[1].x = plot_data["time"]
        fig.data[1].y = plot_data["oxygen_corrected"]

        # Hypoxia threshold
        fig.data[2].x = [plot_data["time"].min(), plot_data["time"].max()]
        fig.data[2].y = [2 / 1.43, 2 / 1.43]

        # Save the updated figure
        st.session_state.plot_fig = fig

    # -----------------------------------------------------
    # Update latest-value displays
    # -----------------------------------------------------

    if not data_values.empty:

        latest = data_values.iloc[-1]

        temperature_display.metric("Temperature", f"{latest['temperature']:.4f} °C")
        oxygen_display.metric("Oxygen", f"{latest['oxygen_corrected']:.4f} mL/L")
        depth_display.metric("Depth", f"{float(latest['depth']):.1f} m")

        st.caption(f"Most recent observation: {latest['time']} UTC")
        st.caption(f"ERDDAP Query: {url}")

        st.plotly_chart(st.session_state.plot_fig,
            use_container_width=True,
            key="china_creek_plot")

# ---------------------------------------------------------
# 2. Dashboard settings
# ---------------------------------------------------------

st.set_page_config(page_title="China Creek Ocean Conditions", layout="wide")

st.title("China Creek Ocean Conditions")

col1, col2, col3 = st.columns(3)

temperature_display = col1.empty()
oxygen_display = col2.empty()
depth_display = col3.empty()

# ---------------------------------------------------------
# 3. Create the plot ONCE
# ---------------------------------------------------------

if "plot_fig" not in st.session_state:

    fig = go.Figure()

    # Temperature trace
    fig.add_trace(
        go.Scatter(
            x=[],
            y=[],
            name="Temperature",
            mode="lines+markers",
            marker=dict(size=4),
            line=dict(color="blue"),
            yaxis="y1",
            showlegend=True
        )
    )

    # Oxygen trace
    fig.add_trace(
        go.Scatter(
            x=[],
            y=[],
            name="Oxygen",
            mode="lines+markers",
            marker=dict(size=4),
            line=dict(color="red"),
            yaxis="y2",
            showlegend=True
        )
    )

    # Hypoxia threshold
    fig.add_trace(
        go.Scatter(
            x=[],
            y=[],
            name="Hypoxia threshold",
            mode="lines",
            line=dict(dash="dash", color="red"),
            yaxis="y2",
            showlegend=True
        )
    )

    # Figure layout
    fig.update_layout(
        title="China Creek O₂ Time Series",
        xaxis=dict(title="Time (UTC)"),
        yaxis=dict(title="Temperature (°C)", side="left"),
        yaxis2=dict(title="Oxygen (mL/L)", side="right", overlaying="y"),
        hovermode="x unified",
        height=550,
        uirevision="china-creek"
    )

    st.session_state.plot_fig = fig

    # Store all observations used by the plot
    st.session_state.plot_data = pd.DataFrame(columns=["time", "temperature", "oxygen_corrected"])

# ---------------------------------------------------------
# 4. Start the dashboard
# ---------------------------------------------------------

update_dashboard()
"""

dashboard_path = Path("china_creek_dashboard.py")
dashboard_path.write_text(dashboard_code)

print(f"Dashboard updated: {dashboard_path.resolve()}")

### Finally, Let's make the dashboard accessible!

So far, we have been viewing the dashboard through `localhost`. This means the
Streamlit server is running on our computer and the browser is connecting to
that same computer.

To allow other computers to access the dashboard, we need
to tell Streamlit to listen for connections from other devices. We do this by
starting Streamlit with the `--server.address` option.

Instead of connecting to:

`http://localhost:8501`

we can connect using the IP address of the computer running the dashboard:

`http://YOUR_IP_ADDRESS:8501`

The IP address identifies the computer running the Streamlit server, while
`8501` identifies the port on which Streamlit is listening.

### Start the dashboard

In terminal, navigate to the folder containing `china_creek_dashboard.py` and run:

```bash
streamlit run china_creek_dashboard.py --server.address 0.0.0.0 
```

The 0.0.0.0 tells streamlit to listen to connections on all network interfaces rather than just localhost. Streamlit will probably return something like 
```bash
Local URL:   http://localhost:8501
Network URL: http://192.168.1.25:8501
```
Other computers should now be able to see your dashboard using the network URL. 
**Note** Only computers on the same local network or wifi as you will be able to see your dashboard. To make this publicly accessible over the internet, you would need to host china_creek_dashboard.py on a cloud-service or server that can constantly execute the command
```bash
streamlit run china_creek_dashboard.py
``` 